# 各ピークの有意度計算スクリプト

全てのピークの有意度を計算し、最小・最大を出す。

## 主要部分

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, vstack
from astropy.modeling.powerlaws import LogParabola1D
from astropy.modeling import models, fitting
from astropy import units as u
from scipy.optimize import curve_fit
from scipy.stats import chi2, norm


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


### ピーク検出・有意度計算

In [2]:
################################################
### # Peak significance calculation - not used
### This must be used with the listing of all the data points
### of consecutive positive deviations, 
### which can be more than three
################################################
def calc_peak_significance(ri):
#     '''
#     fit (array) values for the fit
#     x,y,yerr (arrays) data
#     N total number of points
#     n_free number of parameters we are fitting
#     '''
#     return 1.0/(N-n_free)*sum(((fit - y)/yerr)**2)
# import numpy as np
# from scipy.stats import chi2, norm
# # only those residuals that form the (at least) 3 consecutive positive deviations
# ri = np.array([1, 2, 3.])

# # Chi-square of those residuals
  chi2_value = np.sum(ri**2)

# # p-value for chi-square with m degrees of freedom - m = len(ri) - this is the probability of obtaining a test statistic at least as high as chi2_value
  p_val = 1 - chi2.cdf(chi2_value, df=len(ri))

# # p-value to Gaussian-equivalent one-sided significance
  sigma_equiv = norm.ppf(1 - p_val)

  return p_val, sigma_equiv


###################
# Peak detection
###################
def get_consecutive_bins(s,nconsecutive = 3 ):
  peakinitbin = -1
  for i in range(0,len(s)-nconsecutive+1):
    # print(i)
    if np.all(s[i:i+nconsecutive] > 1.0):
      #   print(i)
      #   print(s[i:i+nconsecutive])
      #   print('---')
      if peakinitbin < 0:
        peakinitbin = i
        # print('Found peak at bin#:',peakinitbin)
        # print(s[peakinitbin:i+nconsecutive])
      if i + nconsecutive == len(s):
        # print('Last bin reached', i, 'for the length ',len(s))
        # print('End of peak at bin #:',i + nconsecutive -1, 'length ', i + nconsecutive - peakinitbin)
        # print(s[peakinitbin:i+nconsecutive])
        # print(peakinitbin, i+ nconsecutive - peakinitbin)
        return peakinitbin, i+ nconsecutive - peakinitbin, calc_peak_significance(s[peakinitbin:i+nconsecutive])
    else:
      if peakinitbin >= 0:
        # print('End of peak at bin#:',i + nconsecutive -2, 'length ', i -1 + nconsecutive - peakinitbin)
        # print(s[peakinitbin:i+nconsecutive ])
        # print(peakinitbin, i+ nconsecutive -1 - peakinitbin)
        return peakinitbin, i+ nconsecutive -1 - peakinitbin, calc_peak_significance(s[peakinitbin:i+nconsecutive-1])
        # if peakbininit[0] > 0:
        #   print('Found longer peak at bin:',peakbininit[0] , 'in total length of ', len(s))
        # else :
        #   print('Longest peak at bin:',i ,'length ', nconsecutive, 'in total length of ', len(s))
  return -1, 0


### 主体関数

In [19]:

nbinsmin=9

###################
# Utility functions
###################
def gaussian(x, amplitude, mu, sigma):
    # return (1.0 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2) : Normal distribution
    return amplitude * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def calc_reduced_chi_square(fit, x, y, yerr, N, n_free):
    '''
    fit (array) values for the fit
    x,y,yerr (arrays) data
    N total number of points
    n_free number of parameters we are fitting
    '''
    return 1.0/(N-n_free)*sum(((fit - y)/yerr)**2)

###################
# Main function
###################
def eval_spectra(filepath,suffix='',sourcename=''): #'data/3C454.3_allsed_14d_min11.ecsv'
  nconsecutive = 3
  sourcenameheader = ''
  if suffix != '':
    sourcenameheader = suffix + '_'

  if sourcename == '':
    sourcename = filepath.split('_')[0]
    sourcename = sourcename.replace('data/', '')
  # logpar_init = LogParabola1D(amplitude=1,x_0=10,alpha=1,beta=1)
  logpar_init = LogParabola1D(
    amplitude = 0.000016494113774149846,
    x_0 = 9.482855296727278,
    alpha = -0.5677548858840729,
    beta=0)
    # beta = 0.1071613245072173)

  ### initialize a linear fitter ###
  # fit = fitting.TRFLSQFitter()
  fit = fitting.DogBoxLSQFitter()

  ### Read the data ###
  t0 = Table.read(filepath)
  nonzero_mask = (t0['e2dnde'] > t0['e2dnde_err'])
  t = t0[nonzero_mask]

  ### MJDごとに処理 ###
  obsdates=np.unique(t['tstart'].data).tolist()
  n_detected_peaks = 0
  array_nbins = []
  array_chisq = []
  t_peakstat = Table(names = ['peak length','SED length','p-value','sigma'], dtype = ['i2','i2','f4' ,'f4'])
  for idx, obsdate in enumerate(obsdates):
    # print(idx, ': obsdate',obsdate)
    mask = (t['tstart']==obsdate)
    x = t[mask]['e_ref']
    y = t[mask]['e2dnde']
    yerr = t[mask]['e2dnde_err']
    nbins = len(x)
    if nbins < nbinsmin:
        print('nbins < nbinsmin:',nbins)
        continue
    array_nbins.append(nbins)
    
    ### fit the data with the fitter ###
    # logpar_init.amplitude.value=x[1]*1.0e-8
    fitted_line = fit(logpar_init, x,y,weights=1.0/yerr, maxiter=200)
    residuals = (y-fitted_line(x))/yerr
    # print(fit.fit_info)
    # residuals = (fitted_line(x)-y)/yerr # negative:dip finding
    reduced_chi_squared = calc_reduced_chi_square(fitted_line(x), x, y, yerr, len(x), 4)
    array_chisq.append(reduced_chi_squared)

    peakbininit = get_consecutive_bins(residuals,nconsecutive)
    if peakbininit[0] > 0:
      if peakbininit[0]+ peakbininit[1] == len(residuals):
        print('Peak reaches the last bin------------------------')
        fitted_line2 = fit(logpar_init, x[0:peakbininit[0]],y[0:peakbininit[0]],weights=1.0/yerr[0:peakbininit[0]], maxiter=200)
        residuals2 = (y-fitted_line2(x))/yerr
        peakbininit2 = get_consecutive_bins(residuals2,nconsecutive)
        print('The significance is ', peakbininit[2])
        print('The significance, after fitting without peak region, is ', peakbininit2[2])
        print('------------------------------------------------')

        
      print('Found peak at bin:',peakbininit[0], 'length ', peakbininit[1], 'in total length of ', len(residuals), 'p-value, sigma:', peakbininit[2])
      t_peakstat.add_row([peakbininit[1], len(residuals), peakbininit[2][0], peakbininit[2][1]])
  print(t_peakstat)
  return t_peakstat

### 実行テスト

In [20]:
# eval_spectra('data/3C454.3_allsed_14d_min11.ecsv')  # 2pos, 0neg/264
# eval_spectra('data/3C454.3_allsed_1d_min11.ecsv')  # 8pos
# eval_spectra('data/BLLac_allsed_14d_min11.ecsv')
# eval_spectra('data/BLLac_allsed_1d_min11.ecsv') 
# eval_spectra('data/3C279_allsed_14d_min11.ecsv') 
# eval_spectra('data/3C279_allsed_1d_min11.ecsv') 
# eval_spectra('data/CTA102_allsed_14d_min11.ecsv') 
# eval_spectra('data/CTA102_allsed_1d_min11.ecsv')
eval_spectra('data/PKS0426-380_allsed_14d_min11.ecsv')


Found peak at bin: 4 length  3 in total length of  17 p-value, sigma: (0.18007202367047193, 0.9150906418819829)
Found peak at bin: 2 length  3 in total length of  16 p-value, sigma: (0.16345042075748928, 0.9803754134993695)
Peak reaches the last bin------------------------
The significance is  (0.003887849993792769, 2.6616572999491352)
The significance, after fitting without peak region, is  (0.001043247802926639, 3.0776360649414833)
------------------------------------------------
Found peak at bin: 11 length  4 in total length of  15 p-value, sigma: (0.003887849993792769, 2.6616572999491352)
nbins < nbinsmin: 7
nbins < nbinsmin: 8
nbins < nbinsmin: 8
Found peak at bin: 12 length  3 in total length of  16 p-value, sigma: (0.039387011852902276, 1.7578441920990209)
Found peak at bin: 3 length  3 in total length of  15 p-value, sigma: (0.17070573755392626, 0.9513800673430649)
nbins < nbinsmin: 8
peak length SED length  p-value     sigma  
----------- ---------- ---------- ---------
     

peak length,SED length,p-value,sigma
int16,int16,float32,float32
3,17,0.18007202,0.9150906
3,16,0.16345042,0.9803754
4,15,0.00388785,2.6616573
3,16,0.03938701,1.7578442
3,15,0.17070574,0.9513801


## 全データ解析
### 天体名対応辞書作成
### - 辞書定義(4FGLカタログ準拠)

In [5]:
from astropy.io import fits
import numpy as np

def get_dict_sourcenames():
  num_sources = 2300
  hdu=fits.open('/Users/kazuma/Workspace/Torun/Fermi/catalog/gll_psc_v35.fit')
  significances = hdu[1].data['Signif_Avg']
  sources = hdu[1].data['Source_Name']
  sources1 = hdu[1].data['ASSOC1']
  sources2 = hdu[1].data['ASSOC2']
  sources_cls=hdu[1].data['CLASS1']
  #Convert source classes to normal array without empty spaces
  source_classes = np.array([entry.strip() for entry in hdu[1].data['CLASS1']])
  #Filter by source class:
  source_classes_selected = np.array(["bll","fsrq","BLL","FSRQ"],dtype='<U5') #see table 5 in https://arxiv.org/pdf/2201.11184
  element_map = np.isin(source_classes, source_classes_selected)
  significances_blazars= significances[element_map]
  sources_blazars = sources[element_map]
  sources_blazars1 = sources1[element_map]
  sources_blazars2 = sources2[element_map]
  sources_blazars_cls = source_classes[element_map]
  #Get index of 20 brightes sources:
  idx = (-significances_blazars).argsort()[:num_sources]
  indices = np.arange(1,num_sources+1)
  # print(f"Number of sources: {len(sources_blazars[idx])}")
  # print(f"Indices of the sources: {indices}")
  #Get the same of the 20 most significant blazars:
  # print(sources_blazars[idx])
  from astropy.table import Table
  # t = Table([sources_blazars[idx],
  #   sources_blazars1[idx],
  #   sources_blazars_cls[idx],
  #   ],names=['4FGL name','assoc name','CLASS','index'])
  

  # sources_blazars_converted  = sources_blazars.strip().replace(' ','_').lower()
  sources_blazars_converted  = sources_blazars.strip().replace('4FGL ','').lower()
  sources_blazars1_converted = sources_blazars1.strip().replace(' ','_').lower()
  sources_blazars_selected_cls = sources_blazars_cls[idx]
  # print(sources_blazars_selected_cls)
  # print(sources_blazars_cls[idx])
  # The elements of sources_blazars_selected_cls are mixture of upper/lower cases. 
  # We unify them to upper case here.
  sources_blazars_selected_cls_converted = np.array([entry.upper() for entry in sources_blazars_selected_cls])
  # print(sources_blazars_selected_cls_converted)  

#   dict_sourcenames = dict(zip(sources_blazars_converted[idx],sources_blazars1_converted[idx]))

  ref_tab_obj = Table([sources_blazars_converted[idx],
  sources_blazars1_converted[idx],
  sources_blazars_selected_cls_converted, #sources_blazars_cls[idx], 
  indices],
  names=['4FGL name','assoc name','CLASS','index'])
# ref_tab_obj[ref_tab_obj['4FGL name'] == '4fgl_j0112.1+2245']
  return ref_tab_obj


### - 辞書生成

In [6]:
dict_sourcename = get_dict_sourcenames()
dict_sourcename[0:2]

4FGL name,assoc name,CLASS,index
str13,str28,str4,int64
j2253.9+1609,3c_454.3,FSRQ,1
j1104.4+3812,mkn_421,BLL,2


### - 天体名書き換え

In [7]:
dict_sourcename[0]['assoc name']  = '3C454.3' 
dict_sourcename[1]['assoc name']  = 'Mrk421' 
dict_sourcename[2]['assoc name']  = 'BLLac' 
dict_sourcename[3]['assoc name']  = 'CTA102' 
dict_sourcename[4]['assoc name']  = '3C279'
dict_sourcename[5]['assoc name']  = 'S50716+71' 
dict_sourcename[6]['assoc name']  = 'PKS1424-41' 
dict_sourcename[7]['assoc name']  = 'PKS0426-380'
dict_sourcename[8]['assoc name']  = 'PKS0537-441'
dict_sourcename[9]['assoc name']  = 'PKS2155-304'
dict_sourcename[10]['assoc name'] = 'PKS0454-234'
dict_sourcename[11]['assoc name'] = 'PKS1510-089'
dict_sourcename[12]['assoc name'] = 'PKS1502+106'
dict_sourcename[13]['assoc name'] = 'TON599'
dict_sourcename[14]['assoc name'] = 'PKS0346-27'
dict_sourcename[15]['assoc name'] = '4c+01.02'
dict_sourcename[16]['assoc name'] = '4c+55.17'
dict_sourcename[17]['assoc name'] = '4c+21.35'
dict_sourcename[18]['assoc name'] = 'PKS1830-211'
dict_sourcename[19]['assoc name'] = 'PKS0208-512'
dict_sourcename[0:15]

4FGL name,assoc name,CLASS,index
str13,str28,str4,int64
j2253.9+1609,3C454.3,FSRQ,1
j1104.4+3812,Mrk421,BLL,2
j2202.7+4216,BLLac,BLL,3
j2232.6+1143,CTA102,FSRQ,4
j1256.1-0547,3C279,FSRQ,5
j0721.9+7120,S50716+71,BLL,6
j1427.9-4206,PKS1424-41,FSRQ,7
j0428.6-3756,PKS0426-380,BLL,8
j0538.8-4405,PKS0537-441,BLL,9


## 実行

### 20天体まとめてピーク解析 14d

In [8]:
n_brightest_sources = 20

table_14d = Table()
for sourceinfo in dict_sourcename[0:n_brightest_sources]:
  print(sourceinfo['assoc name'])
  filepath = 'data/' + sourceinfo['assoc name'].replace('_','').lower() + '_allsed_14d_min11.ecsv'
  t = eval_spectra(filepath)
  if len(t) == 0:
    continue
  t.add_column(np.ones(len(t))*14, name='Days scale')
  t.add_column(np.ones(len(t))*sourceinfo['index'], name='Index')
  t.add_column(np.array([sourceinfo['assoc name']]*len(t)), name='Source Name')
  t.add_column(np.array([sourceinfo['4FGL name']]*len(t)), name='4FGL Name')
  table_14d = vstack([table_14d, t])

print(table_14d)

3C454.3
Found peak at bin: 9 length  3 in total length of  21 p-value, sigma: (0.10888681382133736, 1.2324698360558637)
Found peak at bin: 9 length  3 in total length of  14 p-value, sigma: (0.13022345354587705, 1.1253354640736548)
peak length SED length   p-value     sigma  
----------- ---------- ----------- ---------
          3         21 0.108886816 1.2324698
          3         14  0.13022345 1.1253355
Mrk421
Found peak at bin: 17 length  3 in total length of  21 p-value, sigma: (0.26561617778125846, 0.6261259148765267)
Found peak at bin: 3 length  3 in total length of  12 p-value, sigma: (0.09543889396507621, 1.3079867957977345)
nbins < nbinsmin: 6
Found peak at bin: 7 length  3 in total length of  16 p-value, sigma: (0.023963674699183146, 1.9780120313233365)
nbins < nbinsmin: 8
nbins < nbinsmin: 8
nbins < nbinsmin: 7
Found peak at bin: 4 length  3 in total length of  22 p-value, sigma: (0.003422777657823839, 2.7042655744008415)
Found peak at bin: 6 length  3 in total length of 

    The maximum number of function evaluations is exceeded. [astropy.modeling.fitting]


nbins < nbinsmin: 8
Found peak at bin: 10 length  3 in total length of  19 p-value, sigma: (0.03300192217254705, 1.8383975604742364)
nbins < nbinsmin: 7
nbins < nbinsmin: 8
nbins < nbinsmin: 8
Found peak at bin: 2 length  3 in total length of  18 p-value, sigma: (0.1030282024362229, 1.2644838747842075)
nbins < nbinsmin: 8
nbins < nbinsmin: 8
nbins < nbinsmin: 8
nbins < nbinsmin: 8
nbins < nbinsmin: 6
nbins < nbinsmin: 8
nbins < nbinsmin: 8
nbins < nbinsmin: 7
Found peak at bin: 6 length  3 in total length of  14 p-value, sigma: (0.10174449246173523, 1.271673964141643)
nbins < nbinsmin: 8
peak length SED length   p-value     sigma  
----------- ---------- ----------- ---------
          3         15  0.04325114  1.714144
          3         19 0.033001922 1.8383975
          3         18   0.1030282 1.2644839
          3         14 0.101744495 1.2716739
PKS0454-234
Peak reaches the last bin------------------------
Found peak at bin: 14 length  3 in total length of  17 p-value, sigma: (0

### 20天体まとめてピーク解析 1d

In [9]:
n_brightest_sources = 20

import os.path
table_1d = Table()
for sourceinfo in dict_sourcename[0:n_brightest_sources]:
  print(sourceinfo['assoc name'])
  filepath = 'data/' + sourceinfo['assoc name'].replace('_','').lower() + '_allsed_1d_min11.ecsv'
  if os.path.isfile(filepath) :
    t = eval_spectra(filepath)
    if len(t) == 0:
      continue
    t.add_column(np.ones(len(t))*1, name='Days scale')
    t.add_column(np.ones(len(t))*sourceinfo['index'], name='Index')
    t.add_column(np.array([sourceinfo['assoc name']]*len(t)), name='Source Name')
    t.add_column(np.array([sourceinfo['4FGL name']]*len(t)), name='4FGL Name')
    table_1d = vstack([table_1d, t])
  else:
    print('File not found:', filepath)
print(table_1d)


3C454.3
Found peak at bin: 3 length  3 in total length of  15 p-value, sigma: (0.03516174608807532, 1.8098213742399922)
Peak reaches the last bin------------------------
Found peak at bin: 10 length  3 in total length of  13 p-value, sigma: (0.1336600850148575, 1.1092550506933994)
Found peak at bin: 4 length  3 in total length of  10 p-value, sigma: (0.030220270328241594, 1.877566145675528)
Found peak at bin: 2 length  3 in total length of  11 p-value, sigma: (0.07494361156013984, 1.4399299340574783)
nbins < nbinsmin: 8
nbins < nbinsmin: 8
nbins < nbinsmin: 7
Found peak at bin: 9 length  3 in total length of  17 p-value, sigma: (0.1689914603815259, 0.9581583402545459)
nbins < nbinsmin: 8
Found peak at bin: 5 length  3 in total length of  12 p-value, sigma: (0.11832643331563286, 1.1833944228113904)
nbins < nbinsmin: 8
Found peak at bin: 4 length  3 in total length of  13 p-value, sigma: (0.2232386367752769, 0.7613010500371591)
Found peak at bin: 6 length  3 in total length of  13 p-valu

## 結果解析（table_14d, table_1d）

In [10]:
table_all = Table()
table_all = vstack([table_14d, table_1d])

print(len(table_14d), len(table_1d), len(table_all))
# search the highest and lowest sigma from table_all
table_all['sigma'].max(), table_all['sigma'].min()


63 31 94


(2.7042656, 0.32201186)

In [11]:

# create an array of n strings which are all "hoge"
n=10
array_hoge = np.array(['hoge']*n)
print(array_hoge)



['hoge' 'hoge' 'hoge' 'hoge' 'hoge' 'hoge' 'hoge' 'hoge' 'hoge' 'hoge']


In [12]:
mask = (table_14d['sigma']>2.5)
table_14d[mask]

peak length,SED length,p-value,sigma,Days scale,Index,Source Name,4FGL Name
int16,int16,float32,float32,float64,float64,str11,str12
3,22,0.0034227776,2.7042656,14.0,2.0,Mrk421,j1104.4+3812
4,15,0.00388785,2.6616573,14.0,8.0,PKS0426-380,j0428.6-3756


In [13]:
mask = (table_1d['sigma']>2.5)
table_1d[mask]

peak length,SED length,p-value,sigma,Days scale,Index,Source Name,4FGL Name
int16,int16,float32,float32,float64,float64,str11,str12


In [14]:
mask = (table_14d['Source Name']=='Mrk421')
table_14d[mask]

peak length,SED length,p-value,sigma,Days scale,Index,Source Name,4FGL Name
int16,int16,float32,float32,float64,float64,str11,str12
3,21,0.26561618,0.62612593,14.0,2.0,Mrk421,j1104.4+3812
3,12,0.0954389,1.3079867,14.0,2.0,Mrk421,j1104.4+3812
3,16,0.023963675,1.9780121,14.0,2.0,Mrk421,j1104.4+3812
3,22,0.0034227776,2.7042656,14.0,2.0,Mrk421,j1104.4+3812
3,17,0.12679097,1.1416923,14.0,2.0,Mrk421,j1104.4+3812


In [15]:
mask = (table_14d['Source Name']=='PKS0426-380')
table_14d[mask]

peak length,SED length,p-value,sigma,Days scale,Index,Source Name,4FGL Name
int16,int16,float32,float32,float64,float64,str11,str12
3,17,0.18007202,0.9150906,14.0,8.0,PKS0426-380,j0428.6-3756
3,16,0.16345042,0.9803754,14.0,8.0,PKS0426-380,j0428.6-3756
4,15,0.00388785,2.6616573,14.0,8.0,PKS0426-380,j0428.6-3756
3,16,0.03938701,1.7578442,14.0,8.0,PKS0426-380,j0428.6-3756
3,15,0.17070574,0.9513801,14.0,8.0,PKS0426-380,j0428.6-3756
